# УУНиТ · AI-воркер в Colab

Подготовьте сервер по инструкции `deploy/colab/README.md`. Go, PostgreSQL, RabbitMQ и MinIO остаются
на сервере. Ноутбук запускает Ollama и воркер с локальным HTTP-портом для проверки.

Выберите GPU в настройках среды Colab и пройдите ячейки сверху вниз. Для работы
воркера с внешним приложением потребуется подходящая среда Colab.
[Ограничения и срок жизни среды](https://research.google.com/colaboratory/faq.html).

### 1. Адрес VPS и порты

Измените `VPS_HOST` и `SSH_USER`. Остальные значения должны совпадать с VPS.
Добавьте четыре секрета в боковой панели Colab и разрешите доступ ноутбуку:

- `VPS_SSH_PRIVATE_KEY_B64` — отдельный приватный ключ Colab в base64.
- `VPS_SSH_HOST_KEY` — публичный ключ **сервера** в формате `ssh-ed25519 AAAA…`.
- `VPS_RABBITMQ_PASSWORD` — пароль пользователя RabbitMQ для воркера.
- `VPS_RAG_POSTGRES_PASSWORD` — пароль роли схемы `rag`.

In [ ]:
VPS_HOST = "203.0.113.10"
SSH_USER = "colab-agent"
SSH_PORT = 22

VPS_RABBITMQ_HOST = "10.243.17.11"
VPS_RABBITMQ_PORT = 5672
VPS_POSTGRES_HOST = "10.243.17.10"
VPS_POSTGRES_PORT = 5432
RABBITMQ_USER = "colab_worker"
RABBITMQ_REQUEST_QUEUE = "agent_requests"
RABBITMQ_RESPONSE_QUEUE = "agent_responses"
POSTGRES_DB = "university"
RAG_POSTGRES_USER = "rag_app"

OLLAMA_MODEL = "qwen3:4b"
OLLAMA_EMBEDDING_MODEL = "qwen3-embedding:0.6b"
OLLAMA_NUM_CTX = 8192

LOCAL_RABBITMQ_PORT = 15672
LOCAL_POSTGRES_PORT = 15432
LOCAL_OLLAMA_PORT = 11434
LOCAL_WORKER_PORT = 8000

### 2. Загрузите архив кода

Выберите `colab-worker.tar.gz`, собранный из нужной версии репозитория.
При перезапуске воркера в этой же среде код можно использовать повторно.

In [ ]:
import base64
import json
import os
import re
import shutil
import socket
import subprocess
import sys
import tarfile
import tempfile
import time
from pathlib import Path
from urllib.error import URLError
from urllib.parse import quote
from urllib.request import urlopen

from google.colab import files, userdata

if sys.version_info < (3, 12):
    raise RuntimeError("Выберите в Colab среду с Python 3.12 или новее.")

if "WORK_DIR" not in globals():
    WORK_DIR = Path(tempfile.mkdtemp(prefix="university-colab-", dir="/content"))
APP_DIR = WORK_DIR / "ai-service"
LOG_DIR = WORK_DIR / "logs"
LOG_DIR.mkdir(exist_ok=True)

if not (APP_DIR / "pyproject.toml").is_file():
    uploaded = files.upload()
    archive_name = "colab-worker.tar.gz"
    if archive_name not in uploaded:
        raise ValueError(f"Загрузите файл {archive_name}.")
    archive_path = WORK_DIR / archive_name
    archive_path.write_bytes(uploaded[archive_name])
    del uploaded
    APP_DIR.mkdir(exist_ok=True)
    with tarfile.open(archive_path, "r:gz") as archive:
        archive.extractall(APP_DIR, filter="data")

subprocess.run([sys.executable, "-m", "pip", "install", "-q", str(APP_DIR)], check=True)

if shutil.which("ollama") is None:
    installer = WORK_DIR / "install-ollama.sh"
    subprocess.run(
        ["curl", "-fSL", "--retry", "3", "https://ollama.com/install.sh", "-o", str(installer)],
        check=True,
    )
    subprocess.run(["sh", str(installer)], check=True)

if shutil.which("nvidia-smi") is not None:
    subprocess.run(["nvidia-smi", "-L"], check=False)

jobs = globals().get("jobs", {})


def stop_job(name):
    process = jobs.get(name)
    if process is not None and process.poll() is None:
        process.terminate()
        try:
            process.wait(timeout=15)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait()


def start_job(name, args, *, env=None, cwd=None):
    stop_job(name)
    with (LOG_DIR / f"{name}.log").open("w") as log:
        jobs[name] = subprocess.Popen(
            args, stdin=subprocess.DEVNULL, stdout=log, stderr=subprocess.STDOUT,
            env=env, cwd=cwd,
        )


def wait_until(name, probe, timeout=45):
    until = time.monotonic() + timeout
    while time.monotonic() < until:
        if jobs[name].poll() is not None:
            break
        try:
            if probe():
                return
        except (OSError, URLError):
            pass
        time.sleep(1)
    raise RuntimeError(f"{name} не готов. Откройте ячейку «Диагностика».")


def tcp_ready(port):
    with socket.create_connection(("127.0.0.1", port), timeout=2):
        return True


def http_ready(port, path):
    with urlopen(f"http://127.0.0.1:{port}{path}", timeout=2) as response:
        return response.status == 200

### 3. Подключите туннель к VPS

OpenSSH проверяет ключ хоста и перенаправляет только два порта на локальный интерфейс.
После сбоя подключения повторите эту ячейку, затем ячейку запуска воркера.

In [ ]:
if VPS_HOST == "203.0.113.10" or not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9.-]*", VPS_HOST):
    raise ValueError("Укажите IPv4 или DNS-имя VPS.")
if not re.fullmatch(r"[A-Za-z0-9_][A-Za-z0-9_.-]*", SSH_USER):
    raise ValueError("Проверьте SSH_USER.")

key_path = WORK_DIR / "ssh_key"
key_path.touch(mode=0o600, exist_ok=True)
key_path.chmod(0o600)
private_key = base64.b64decode(userdata.get("VPS_SSH_PRIVATE_KEY_B64").strip(), validate=True)
key_path.write_bytes(private_key)
del private_key

host_key = userdata.get("VPS_SSH_HOST_KEY").split()
if len(host_key) < 2 or host_key[0] != "ssh-ed25519" or not re.fullmatch(r"[A-Za-z0-9+/=]+", host_key[1]):
    raise ValueError("Укажите публичный ключ хоста формата ssh-ed25519.")
host_entry = VPS_HOST if SSH_PORT == 22 else f"[{VPS_HOST}]:{SSH_PORT}"
known_hosts = WORK_DIR / "known_hosts"
known_hosts.write_text(f"{host_entry} {' '.join(host_key[:2])}\n")

start_job("ssh", [
    "ssh", "-F", "/dev/null", "-N", "-T", "-i", str(key_path),
    "-p", str(SSH_PORT), "-l", SSH_USER,
    "-o", "BatchMode=yes", "-o", "IdentitiesOnly=yes", "-o", "ExitOnForwardFailure=yes",
    "-o", "StrictHostKeyChecking=yes", "-o", f"UserKnownHostsFile={known_hosts}",
    "-o", "HostKeyAlgorithms=ssh-ed25519",
    "-o", "ConnectTimeout=10", "-o", "ServerAliveInterval=20", "-o", "ServerAliveCountMax=3",
    "-L", f"127.0.0.1:{LOCAL_RABBITMQ_PORT}:{VPS_RABBITMQ_HOST}:{VPS_RABBITMQ_PORT}",
    "-L", f"127.0.0.1:{LOCAL_POSTGRES_PORT}:{VPS_POSTGRES_HOST}:{VPS_POSTGRES_PORT}",
    "--", VPS_HOST,
])
wait_until("ssh", lambda: tcp_ready(LOCAL_RABBITMQ_PORT) and tcp_ready(LOCAL_POSTGRES_PORT))

worker_env = {
    **os.environ,
    "RABBITMQ_URL": (
        f"amqp://{quote(RABBITMQ_USER, safe='')}:"
        f"{quote(userdata.get('VPS_RABBITMQ_PASSWORD'), safe='')}@127.0.0.1:{LOCAL_RABBITMQ_PORT}/"
        "?heartbeat=30"
    ),
    "RABBITMQ_QUEUE": RABBITMQ_REQUEST_QUEUE,
    "RABBITMQ_RESPONSE_QUEUE": RABBITMQ_RESPONSE_QUEUE,
    "RAG_POSTGRES_HOST": "127.0.0.1",
    "RAG_POSTGRES_PORT": str(LOCAL_POSTGRES_PORT),
    "RAG_POSTGRES_DB": POSTGRES_DB,
    "RAG_POSTGRES_USER": RAG_POSTGRES_USER,
    "RAG_POSTGRES_PASSWORD": userdata.get("VPS_RAG_POSTGRES_PASSWORD"),
    "OLLAMA_BASE_URL": f"http://127.0.0.1:{LOCAL_OLLAMA_PORT}",
    "OLLAMA_MODEL": OLLAMA_MODEL,
    "OLLAMA_EMBEDDING_MODEL": OLLAMA_EMBEDDING_MODEL,
    "OLLAMA_NUM_CTX": str(OLLAMA_NUM_CTX),
}

subprocess.run(
    [sys.executable, "-m", "app.index_campus", "--import-only"],
    cwd=APP_DIR, env=worker_env, check=True,
)
print("Туннель поднят. PostgreSQL принимает подключение, миграции применены.")

### 4. Ollama и индексация

Модели сохраняются в текущей среде Colab. Векторы сохраняются в PostgreSQL на VPS.
Повторный запуск пропускает уже индексированные записи.

In [ ]:
ollama_env = {**os.environ, "OLLAMA_HOST": f"127.0.0.1:{LOCAL_OLLAMA_PORT}"}
if "ollama" not in jobs or jobs["ollama"].poll() is not None:
    start_job("ollama", ["ollama", "serve"], env=ollama_env)
wait_until("ollama", lambda: http_ready(LOCAL_OLLAMA_PORT, "/api/tags"))

for model in (OLLAMA_MODEL, OLLAMA_EMBEDDING_MODEL):
    subprocess.run(["ollama", "pull", model], env=ollama_env, check=True)

subprocess.run(
    [sys.executable, "-m", "app.index_campus"],
    cwd=APP_DIR, env=worker_env, check=True,
)

### 5. Запуск AI-воркера

При успехе `/health` показывает готовность RabbitMQ и Ollama. Теперь можно отправить
запрос через Go API на VPS. Эту ячейку можно повторить для перезапуска воркера.

In [ ]:
start_job(
    "worker",
    [sys.executable, "-m", "uvicorn", "app.main:app", "--host", "127.0.0.1",
     "--port", str(LOCAL_WORKER_PORT)],
    env=worker_env, cwd=APP_DIR,
)
wait_until("worker", lambda: http_ready(LOCAL_WORKER_PORT, "/health"))
with urlopen(f"http://127.0.0.1:{LOCAL_WORKER_PORT}/health", timeout=3) as health:
    print(json.load(health))

### Диагностика и остановка

Выведите последние записи нужного процесса. При ошибке SSH проверьте ключ хоста, права
на выделенный ключ и разрешённые порты. Ошибки авторизации RabbitMQ будут в журнале воркера.

In [ ]:
name = "worker"  # ssh, ollama или worker
log_file = LOG_DIR / f"{name}.log"
if log_file.is_file():
    print("\n".join(log_file.read_text(errors="replace").splitlines()[-40:]))

In [ ]:
# По окончании сеанса измените значение на True и выполните эту ячейку.
STOP_WORKER = False
if STOP_WORKER:
    for name in ("worker", "ollama", "ssh"):
        stop_job(name)
    if "key_path" in globals():
        key_path.unlink(missing_ok=True)
    print("Процессы Colab остановлены.")